## disease_distribution_colab
- Role: check the disease classes and how they are distributed.
- I focused on label balance after the split.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

raw_path = Path('disease_symptoms_raw.csv')
if not raw_path.exists():
    raw_path = Path('data/raw/disease_symptoms_raw.csv')

print('raw_path:', raw_path)
raw_df = pd.read_csv(raw_path)
raw_df.head()


### Clean columns
- I clean the column names before splitting the file.


In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    drop_cols = [c for c in df.columns if c == '' or c.lower().startswith('unnamed')]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

raw_df = clean_columns(raw_df)
target = 'prognosis'
symptom_cols = [c for c in raw_df.columns if c != target]
print('Raw shape:', raw_df.shape)


### Split raw data
- I split by unique symptom pattern so the same pattern does not appear in both train and test.


In [ ]:
from sklearn.model_selection import train_test_split

raw_df['_symptom_signature'] = raw_df[symptom_cols].astype(str).agg('|'.join, axis=1)
unique_patterns = raw_df[[target, '_symptom_signature']].drop_duplicates()

test_signatures = []
for label, group in unique_patterns.groupby(target):
    _, label_test = train_test_split(
        group['_symptom_signature'],
        test_size=0.2,
        random_state=42,
    )
    test_signatures.extend(label_test.tolist())

test_signature_set = set(test_signatures)
train_df = raw_df[~raw_df['_symptom_signature'].isin(test_signature_set)].copy()
test_df = raw_df[raw_df['_symptom_signature'].isin(test_signature_set)].copy()

output_dir = Path('data')
output_dir.mkdir(exist_ok=True)
train_output = output_dir / 'Training.csv'
test_output = output_dir / 'Testing.csv'

train_df = train_df.drop(columns=['_symptom_signature'])
test_df = test_df.drop(columns=['_symptom_signature'])
train_df.to_csv(train_output, index=False)
test_df.to_csv(test_output, index=False)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Saved:', train_output)
print('Saved:', test_output)


### Leakage check
- I check that train and test do not share the same symptom pattern.


In [ ]:
train_signatures = set(train_df[symptom_cols].astype(str).agg('|'.join, axis=1))
test_signatures = set(test_df[symptom_cols].astype(str).agg('|'.join, axis=1))
print('Shared symptom patterns:', len(train_signatures & test_signatures))


### Disease counts in raw data
- I looked at how many rows belong to each disease.


In [ ]:
raw_counts = raw_df[target].value_counts()
display(raw_counts.head(10))

plt.figure(figsize=(10, 4))
raw_counts.head(10).plot(kind='bar')
plt.title('Top 10 diseases in raw data')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### Train and test balance
- I compared the class balance after the split.


In [ ]:
train_counts = train_df[target].value_counts()
test_counts = test_df[target].value_counts()
print('Train min/max:', train_counts.min(), train_counts.max())
print('Test min/max:', test_counts.min(), test_counts.max())
display(train_counts.head(10))
display(test_counts.head(10))


### Short findings
- The dataset contains 41 disease labels.
- The raw data is close to balanced, but not perfectly equal.
- The new split still keeps every disease in both train and test.
- This is useful for a fairer baseline check.
